This script reads the .csv files provided by Equinor and creates new .csv files without duplicates and with additional failure info.

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import scipy as sp
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import datetime
import matplotlib.dates as mdates
import re
import ESP_funcs as esp

import utility

# Load data and remove downtime

#### Load operational data


The provided data includes timestamps where 'Well_down' == True ('VSD power frequency' < 7 Hz). We remove these rows from the data to reduce memory usage, as their values are not passed to the pipeline.

In [2]:
# Reading hourly data removing when well is down

hour_source = 'EPIC/Dados/full_sensor_1h [old].csv'
min_source = 'EPIC/Dados/full_sensor_1min_resampled_V2 [old].csv'


data_hour = utility.clean_database(hour_source, 'Well_down', 0, drop_column=True, comparison='=')
data_min = utility.clean_database(min_source, 'Well_down', 0, drop_column=True, comparison='=')


In [3]:
# Merging both to one DataFrame
data_full = pd.merge(data_hour,data_min,on=['time','Well Run'],how='outer')
data_full.set_index('time',inplace=True)
data_full.index = pd.to_datetime(data_full.index)
data_full.index = data_full.index.tz_localize(None)

del(data_min)
del(data_hour)


#### Load failure table

In [ ]:
# Reading failure table

failure_name = "EPIC/Dados/Peregrino ESP Run Life 240410 [SMALL].xlsx"
data_failure = pd.read_excel(failure_name,skiprows=2)

# Função de substituição
def substituir(match):
    return f" {match.group(1)}"

# Substituição usando expressão regular
for dat in data_failure.index:
    data_failure.loc[dat,'Well Name'] = re.sub(r'\(#(\d+)\)', substituir, data_failure.loc[dat,'Well Name'])
    if len(data_failure.loc[dat,'Well Name']) in [4,5]:
        data_failure.loc[dat,'Well Name'] = f"{data_failure.loc[dat,'Well Name']} 1"

data_failure.set_index('Well Name',inplace=True)

name = data_failure.index

wells = data_full['Well Run'].unique()



#### Removing repeated data

Some wells have repeated data. See the example below:

In [ ]:
# Repeated data example

data_A = data_full[data_full['Well Run']== 'A-09 1']
data_B = data_full[data_full['Well Run']== 'A-09 2']

a = data_A['VSD power frequency']
b = data_B['VSD power frequency']

data_A['VSD power frequency'].iloc[8815:8850].plot(style='o')

Well A-09 1 has a failure date of 4 of December 2013, and there is indeed an interruption on that date. However, after a few months, operation resumes. What is this data that appeared after failure?

In [ ]:
plt.figure()
a.iloc[8817::].plot(style='r.-')
b.plot(style='y--')

If we compare this extra data to the data from well run A-09 2, we see that they are the exact same! In short, some well runs have duplicate data after failure. Thus, we have to do something to remove these duplicates.

All data at least 24 hours after failure date is thus discarded.

In [28]:
fail_wells = []
for well in data_full['Well Run'].unique():

    # Getting failure date for a specific well
    ind = data_failure.index==well
    fail_date = data_failure[ind]['Failure Date'].values

    # Testing if failure date is before the end date
    ind_well = data_full['Well Run'] == well
    dates_well = ind_well[ind_well].index
    end_date = dates_well.max()
    if bool(data_failure[ind]['ESP Health'].values) is False:
        a = fail_date[0] - dates_well

        if a.min().days==0:
            a = a - datetime.timedelta(seconds=a.min().seconds)
        elif a.min().days == -1:
            a = a - datetime.timedelta(seconds=a.min().seconds, days=-1)

        data_full.loc[ind_well, 'Failure distance'] = a

        fail_wells.append(well)
    else:
        data_full.loc[ind_well, 'Failure distance'] = pd.NA
        
data_full = data_full[~(data_full['Failure distance'].dt.components.days < 0)]

#### Saving the data



After this pre-processing, the data is saved again, under one file

In [ ]:
out_name = 'EPIC/Dados/merged_data.csv'
data_full.to_csv(out_name,index=True)